# SHIPP Stage 17 — ORS Failure / Retry Validation

**Purpose:** prove the production ORS client retries retryable failures, honors `Retry-After`, preserves error/raw evidence, and never leaks the API key.

This is a **deterministic failure test**. It does not consume ORS quota and does not depend on an external outage. The real ORS raw-response path is validated separately by Task 92.


In [0]:
%run ../common/shipp_silver_lib

In [0]:
import json
import uuid
from datetime import datetime, timezone

import requests

from data_pipeline.ingestion.ors_client import ORSClient


class FakeResponse:
    def __init__(self, status, body, headers=None):
        self.status_code = status
        self.text = body
        self.headers = headers or {}


class FakeSession:
    def __init__(self, responses):
        self.responses = list(responses)
        self.calls = []

    def post(self, url, data=None, headers=None, timeout=None):
        self.calls.append({'url': url, 'data': data, 'headers': headers})
        item = self.responses.pop(0)
        if isinstance(item, Exception):
            raise item
        return item


OK_BODY = json.dumps({'distances': [[6.4]], 'durations': [[840.0]]})
sleep_calls = []
retry_session = FakeSession([
    FakeResponse(429, 'rate limited', {'Retry-After': '3'}),
    FakeResponse(200, OK_BODY),
])
client = ORSClient(
    'validation-key-never-persisted',
    session=retry_session,
    sleep=sleep_calls.append,
)

retry_result = client.matrix_many_to_one(
    [(24.49, 54.40)],
    (24.50, 54.38),
)

assert retry_result.ok
assert retry_result.attempts == 2
assert sleep_calls == [3.0]
assert 'validation-key-never-persisted' not in retry_result.request_json
assert retry_result.response_body == OK_BODY

failure_session = FakeSession([
    requests.ConnectionError('controlled outage'),
    requests.ConnectionError('controlled outage'),
    requests.ConnectionError('controlled outage'),
])
failure_sleeps = []
failure_client = ORSClient(
    'validation-key-never-persisted',
    session=failure_session,
    sleep=failure_sleeps.append,
    max_retries=2,
)
failure_result = failure_client.matrix_many_to_one(
    [(24.49, 54.40)],
    (24.50, 54.38),
)

assert not failure_result.ok
assert failure_result.attempts == 3
assert len(failure_sleeps) == 2
assert failure_result.http_status is None
assert 'ConnectionError' in (failure_result.error_message or '')
assert 'validation-key-never-persisted' not in failure_result.request_json

print('PASS — retryable 429 retried and Retry-After honored.')
print('PASS — network failure exhausted retries without raising.')
print('PASS — request evidence contains no API key.')


In [0]:
RUN_ID = str(uuid.uuid4())
CHECKED_AT = datetime.now(timezone.utc).replace(tzinfo=None)
TARGET = 'bootcamp_students.shipp_gold.ors_failure_validation_log'

row = [(
    RUN_ID,
    'PASS',
    retry_result.attempts,
    float(sleep_calls[0]),
    failure_result.attempts,
    failure_result.error_message,
    CHECKED_AT,
)]
schema = '''
validation_run_id string,
overall_status string,
retry_attempts int,
retry_after_seconds double,
failure_attempts int,
failure_error_message string,
checked_at timestamp
'''

spark.createDataFrame(row, schema).write.format('delta').mode('append').saveAsTable(TARGET)
display(spark.table(TARGET).filter(f"validation_run_id = '{RUN_ID}'"))
dbutils.notebook.exit(json.dumps({
    'validation_run_id': RUN_ID,
    'status': 'PASS',
    'retry_attempts': retry_result.attempts,
    'failure_attempts': failure_result.attempts,
}, default=str))
